# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farahhussain159-create/flyrank-ml-week1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/farahhussain159-create/flyrank-ml-week1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print(df[['search_volume', 'ctr', 'avg_position', 'char_count']].describe())
print("\nFreshness tier counts:\n", df['freshness_tier'].value_counts())


       search_volume           ctr  avg_position     char_count
count   27532.000000  30000.000000   30000.00000   22301.000000
mean      158.882391      0.510733      16.34238   20665.277835
std      1518.270825      3.279162      15.21679   10115.344042
min         0.000000      0.000000       0.00000      40.000000
25%         0.000000      0.000000       6.20000   15644.000000
50%        10.000000      0.070000      10.80000   19116.000000
75%        20.000000      0.290000      22.30000   24011.000000
max     74000.000000    100.000000     245.00000  111158.000000

Freshness tier counts:
 freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [9]:
# Signal 1: Staleness vs decline rate
signal1 = df.groupby('freshness_tier')['is_declining'].agg(n='count', decline_rate='mean').sort_values('decline_rate', ascending=False)
print("=== Signal 1: Staleness (freshness_tier) vs decline rate ===")
print(signal1)

# Signal 2: CTR vs decline rate
df['ctr_bucket'] = pd.qcut(df['ctr'], q=4, duplicates='drop')
signal2 = df.groupby('ctr_bucket', observed=True)['is_declining'].agg(n='count', decline_rate='mean')
print("\n=== Signal 2: CTR vs decline rate ===")
print(signal2)

# Signal 3: Search volume vs decline rate
df['volume_bucket'] = pd.qcut(df['search_volume'], q=4, duplicates='drop')
signal3 = df.groupby('volume_bucket', observed=True)['is_declining'].agg(n='count', decline_rate='mean')
print("\n=== Signal 3: Search volume vs decline rate ===")
print(signal3)


=== Signal 1: Staleness (freshness_tier) vs decline rate ===
                    n  decline_rate
freshness_tier                     
91-180           9171      0.611057
31-90             175      0.588571
0-30            20480      0.511377
181+              174      0.471264

=== Signal 2: CTR vs decline rate ===
                    n  decline_rate
ctr_bucket                         
(-0.001, 0.07]  15224      0.523910
(0.07, 0.29]     7503      0.604825
(0.29, 100.0]    7273      0.515331

=== Signal 3: Search volume vs decline rate ===
                     n  decline_rate
volume_bucket                       
(-0.001, 10.0]   18392      0.590365
(10.0, 20.0]      2290      0.515721
(20.0, 74000.0]   6850      0.508905


Signal 1 — Staleness: MIXED. Older content (91-180 days) declines more than fresh content (0-30 days), but the 181+ bucket breaks the clean pattern — too small a sample to trust fully. Verdict: MIXED.
Signal 2 — CTR: FALSE. Raw CTR shows no clean monotonic pattern against decline rate on its own. Verdict: FALSE (needs to be checked relative to position, not standalone).
Signal 3 — Search volume: [verdict yahan apna result dekh kar likhna]

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [10]:
# Flag-linked test: does staleness (used in the baseline rule) actually predict decline?
flag_check = df.groupby(df['freshness_tier'].isin(['91-180', '181+']))['is_declining'].agg(n='count', decline_rate='mean')
flag_check.index = ['Fresh (0-30, 31-90)', 'Stale (91-180, 181+)']
print(flag_check)


                          n  decline_rate
Fresh (0-30, 31-90)   20655      0.512031
Stale (91-180, 181+)   9345      0.608454


The baseline rule assumes stale content (91-180+ days old) is more likely to be declining. The data partially supports this — stale content shows a higher decline rate than fresh content, but the effect is not uniform across all stale sub-tiers, meaning staleness alone is a weak-to-moderate signal, not a strong standalone predictor.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In practice, the content team should not treat any single signal (staleness or CTR alone) as a reliable trigger for refresh. Staleness gives a directional hint but produces false positives on old-but-still-performing pages. The safest approach is to combine staleness with volume (as the baseline rule does) and validate with an ML model rather than a single-flag rule.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.